# 🟢 LEARN B — 시간축과 결측값 판단

**목표:** 깨끗하게 읽힌 표를 실제 **시계열 데이터**로 만든다.

핵심은 "결측치니까 다 평균으로 채운다"가 아니다.
**무슨 열인지에 따라 처리 정책을 달리한다.**

In [ ]:
import numpy as np
import pandas as pd

PATH = "data/AirQuality_DIRTY_LEARN.csv"

df = pd.read_csv(PATH, sep=';')
df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')

numeric_cols = [
    'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)',
    'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)',
    'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH'
]

df[numeric_cols] = df[numeric_cols].replace(['?', 'NA', 'error'], np.nan)
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df[numeric_cols] = df[numeric_cols].replace(-200, np.nan)

df.head()

## B1. 날짜와 시간 합치기

`Date`, `Time`이 문자열로 따로 들어 있다.

먼저 공백을 제거:

```python
df['Date'] = df['Date'].str.strip()
df['Time'] = df['Time'].str.strip()
```

그리고:

```python
pd.to_datetime(..., format='%d/%m/%Y %H:%M:%S', errors='coerce')
```

를 이용해 `datetime` 열 하나로 합쳐라.

일부러 잘못된 날짜도 몇 개 있으므로 `errors='coerce'`가 중요하다.

변환 후 `datetime`이 NaT인 행의 개수를 확인해라.

In [ ]:
# TODO B1

## B2. 시간 순서와 중복

RAW 행 순서는 섞여 있고 중복 행도 있다.

해야 할 순서:

1. invalid datetime 행 제거
2. 완전히 동일한 중복 행 제거
3. 같은 `datetime`이 중복이라면 하나만 남기기
4. `datetime` 오름차순 정렬
5. index 재설정

새 문법 예:

```python
df.drop_duplicates(...)
df.sort_values(...)
df.reset_index(drop=True)
```

끝난 뒤 아래 조건이 모두 True인지 검사해라.

```python
df['datetime'].is_monotonic_increasing
df['datetime'].is_unique
```

In [ ]:
# TODO B2

## B3. 모든 NaN을 똑같이 채우면 안 되는 이유

다음 두 종류를 나눈다.

### Ground Truth
인증된 분석기로 측정한 목표값:

```python
gt_cols = ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']
```

측정 자체가 없었던 값을 임의로 만들어 내면 분석 결과가 왜곡될 수 있으므로
**이번 챌린지에서는 그대로 NaN으로 둔다.**

### Sensor + weather
시간에 따라 연속적으로 측정되는 센서/기상값:

```python
continuous_cols = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
    'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH'
]
```

이번 연습에서는 시간 순서로 정렬된 상태에서 선형 보간:

```python
df[continuous_cols] = df[continuous_cols].interpolate(
    method='linear',
    limit_direction='both'
)
```

을 사용한다.

보간 전후 결측 개수를 비교해라.

In [ ]:
gt_cols = ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']

continuous_cols = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
    'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH'
]

# TODO B3

## B4. 벡터화 파생 변수

반복문 없이 세 열을 만든다.

### ① sensor_mean
5개 센서 응답의 행별 평균.

```python
sensor_cols = [...]
df['sensor_mean'] = df[sensor_cols].mean(axis=1)
```

### ② is_rush_hour
시간대가 07~09시 또는 17~19시면 1, 아니면 0.

반드시 `np.where()` 사용.

### ③ temp_z
온도 `T`를 z-score로 표준화.

\[
z = \frac{x-\mu}{\sigma}
\]

NumPy/Pandas 벡터 연산으로 한 번에 계산.

In [ ]:
sensor_cols = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
    'PT08.S4(NO2)', 'PT08.S5(O3)'
]

# TODO B4

## ✅ LEARN B 체크포인트

아래를 전부 확인해라.

- `datetime` dtype이 datetime인가?
- 시간 오름차순인가?
- datetime 중복이 없는가?
- continuous_cols에 NaN이 남지 않았는가?
- gt_cols에는 NaN이 남아 있어도 되는가?
- `sensor_mean`, `is_rush_hour`, `temp_z`가 생성됐는가?
- `temp_z`의 평균이 거의 0인가?

In [ ]:
# TODO CHECKPOINT